# Notebook 6 (optional): rerun the selection with meaning-based feature coding

Answers second-round concern B. In the paper, the search's noise objective used the range-based coding, which perturbs Protocol Type and Duration (TTL) and holds Variance fixed. This notebook repeats the ten-run search with the **meaning-based coding** driving the noise objective. It then selects the consensus subset, the single three-objective solution, and size-matched MI and RFE subsets, and evaluates them exactly as in Notebook 4.

Run it after Notebook 4 has finished, in the `Python (iomt)` kernel, from the same folder. It takes about 1 hour (about 5 minutes per run) and saves progress after every run, so you can stop and restart it. Results go to `results_revision/meaning_search_results.xlsx`.

In [1]:
# Load and run every code cell of Notebook 4 in this notebook, so its functions and data are available here.
# Its main loop skips runs already saved, so this only reloads data and rebuilds the perturbed test copies.
# (This replaces the Jupyter run command, which needs the nbformat package.)
import json as _json
_nb4 = _json.load(open("4_Supplementary_Analyses.ipynb", encoding="utf-8"))
for _cell in _nb4["cells"]:
    if _cell["cell_type"] == "code":
        _src = "".join(_cell["source"]) if isinstance(_cell["source"], list) else _cell["source"]
        exec(compile(_src, "4_Supplementary_Analyses.ipynb", "exec"), globals())
print("Notebook 4 loaded.")
REV_OUT = "results_revision"; os.makedirs(REV_OUT, exist_ok=True)
P = "Proposed: NSGA-II (3-obj) + consensus"
assert len(state["runs"]) == N_RUNS, "Notebook 4 must finish all runs first"

def pick_single(F, X, n_obj):   # same rule as Notebook 3
    val_f1 = 1 - F[:, 0]
    ok = np.where(val_f1 >= val_f1.max() - F1_TOLERANCE)[0]
    tie = F[ok, 2] if n_obj == 3 else np.zeros(len(ok))
    best = ok[np.lexsort((tie, F[ok, 1]))[0]]
    return SEL_COLS[np.where(X[best])[0]]

def mi_select(X_tr, y_tr, k, seed):   # same as Notebook 3
    idx = np.random.default_rng(seed).choice(len(X_tr), min(20000, len(X_tr)), replace=False)
    scores = mutual_info_classif(X_tr[idx][:, SEL_COLS], y_tr[idx], random_state=seed)
    return SEL_COLS[np.argsort(scores)[::-1][:k]]

def rfe_select(X_tr, y_tr, k, seed):  # same as Notebook 3
    idx = np.random.default_rng(seed).choice(len(X_tr), min(10000, len(X_tr)), replace=False)
    rfe = RFE(make_forest(30, seed), n_features_to_select=k, step=2).fit(X_tr[idx][:, SEL_COLS], y_tr[idx])
    return SEL_COLS[np.where(rfe.support_)[0]]

def paired(a, b):
    d = np.asarray(a, float) - np.asarray(b, float); n = len(d)
    h = stats.t.ppf(0.975, n - 1) * d.std(ddof=1) / np.sqrt(n)
    p = stats.wilcoxon(d).pvalue if np.any(d != 0) else 1.0
    return {"mean diff": d.mean(), "95% CI low": d.mean() - h, "95% CI high": d.mean() + h,
            "runs favoring first": int((d > 0).sum()), "Wilcoxon p": p}
print("Range-coded perturbed features:", int((~INDICATOR).sum()), "| meaning-coded:", int((~SEMANTIC_FIXED).sum()))

Using checkpoint: results_revised\checkpoint.pkl
Windows-10-10.0.26300-SP0 | CPU: Intel64 Family 6 Model 142 Stepping 10, GenuineIntel | cores: 8 | Python 3.11.17 | scikit-learn 1.9.1 | pymoo 0.6.2 | numpy 2.4.6 | pandas 3.0.6
Runs in checkpoint: 10
Range-coded perturbed features: 20 | meaning-coded perturbed features: 19
Perturbed under range coding but not meaning coding: ['Protocol Type', 'Duration']
Perturbed under meaning coding but not range coding: ['Variance']
Perturbation sets: ['gauss_5', 'gauss_10', 'gauss_20', 'semantic_gauss_10', 'uniform', 'laplace', 'missing_10', 'missing_20', 'quantise']
Functions ready.
Saved progress found for runs: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
{'max_abs_diff_clean_F1': np.float64(0.0), 'max_abs_diff_F1_noise10': np.float64(0.0), 'proposed_subset_reproduced_runs': 10, 'runs': 10}
{'mean difference': np.float64(-0.00495226344328078), '90% CI low': np.float64(-0.008142615247464569), '90% CI high': np.float64(-0.0017619116390969916), 'margin': 0.01, 'W

In [2]:
SAVE6 = os.path.join(REV_OUT, "meaning_search_checkpoint.pkl")
st6 = {}
if os.path.exists(SAVE6):
    with open(SAVE6, "rb") as fh: st6 = pickle.load(fh)
    print("Saved progress found for runs:", sorted(st6))

INDICATOR_RANGE = INDICATOR.copy()
INDICATOR = SEMANTIC_FIXED.copy()        # the meaning-based mask now drives the noise objective in make_folds
try:
    for run in range(N_RUNS):
        if run in st6: continue
        t0 = time.time(); X_tr, y_tr = sample_train(run); folds = make_folds(X_tr, y_tr, run)
        res, aX, aF, hv = run_search(folds, run, 3)
        cons, n_near = pick_consensus(aF, aX, F1_TOLERANCE, CONSENSUS)
        single = pick_single(np.atleast_2d(res.F), np.atleast_2d(res.X), 3)
        k = len(cons)
        subsets = {"Proposed (meaning-coded search)": cons,
                   "NSGA-II (3-obj), meaning-coded search": single,
                   "MI (top-k), k matched": mi_select(X_tr, y_tr, k, run),
                   "RFE (top-k), k matched": rfe_select(X_tr, y_tr, k, run)}
        rows = []
        for mth, cols in subsets.items():
            r, _ = evaluate_full(cols, X_tr, y_tr, run, extended=False)
            rows.append({"run": run, "method": mth, **r})
        np.savez_compressed(os.path.join(REV_OUT, f"archive_meaning_run{run + 1}.npz"), X=aX, F=aF, hv=np.array(hv))
        st6[run] = {"rows": rows, "n_near": n_near, "hv": hv,
                    "subsets": {m: [FEATURES_ALL[i] for i in v] for m, v in subsets.items()}}
        with open(SAVE6, "wb") as fh: pickle.dump(st6, fh)
        print(f"run {run + 1}/{N_RUNS} done in {(time.time() - t0) / 60:.1f} min | consensus size {k}", flush=True)
finally:
    INDICATOR = INDICATOR_RANGE          # restore the original coding

run 1/10 done in 3.4 min | consensus size 22
run 2/10 done in 3.1 min | consensus size 18
run 3/10 done in 5.1 min | consensus size 16
run 4/10 done in 4.5 min | consensus size 26
run 5/10 done in 3.5 min | consensus size 12
run 6/10 done in 3.4 min | consensus size 24
run 7/10 done in 4.1 min | consensus size 20
run 8/10 done in 3.4 min | consensus size 19
run 9/10 done in 4.5 min | consensus size 18
run 10/10 done in 4.2 min | consensus size 14


In [3]:
rows6 = pd.DataFrame([r for R in st6.values() for r in R["rows"]])
orig = pd.DataFrame([r for R in state["runs"].values() for r in R["rows"]])
orig = orig[orig.method.isin([P, "NSGA-II (3-obj)", "All features"])].copy()
orig["method"] = orig["method"].replace({P: "Proposed (range-coded search, paper)", "NSGA-II (3-obj)": "NSGA-II (3-obj), range-coded search (paper)"})
allr = pd.concat([rows6, orig[rows6.columns.intersection(orig.columns)]], ignore_index=True)
cols_m = ["n_features", "macro_F1", "F1_gauss_10", "F1_semantic_gauss_10"]
M6 = allr.groupby("method")[cols_m].mean().sort_values("F1_semantic_gauss_10", ascending=False)
print(M6.round(4))

PM = "Proposed (meaning-coded search)"
pc = []
for metric in ["F1_semantic_gauss_10", "F1_gauss_10", "macro_F1"]:
    w = allr.pivot(index="run", columns="method", values=metric)
    for c in w.columns:
        if c != PM: pc.append({"metric": metric, "vs": c, **paired(w[PM], w[c])})
PC6 = pd.DataFrame(pc); print(PC6.round(4).to_string(index=False))

new_sets = [set(st6[r]["subsets"][PM]) for r in sorted(st6)]
old_sets = [set(state["runs"][r]["subsets"][P]) for r in sorted(st6)]
overlap = [len(a & b) / len(a | b) for a, b in zip(new_sets, old_sets)]
S6 = pd.DataFrame({"quantity": ["Nogueira phi, meaning-coded consensus", "Nogueira phi, range-coded consensus (paper)",
                                "mean Jaccard overlap, meaning vs range consensus (same run)",
                                "mean consensus size, meaning-coded", "mean near-optimal subsets per run"],
                   "value": [nogueira(new_sets), nogueira(old_sets), float(np.mean(overlap)),
                             float(np.mean([len(s) for s in new_sets])), float(np.mean([st6[r]["n_near"] for r in st6]))]})
freq = pd.Series([f for s in new_sets for f in s]).value_counts() / len(new_sets)
print(S6.round(4).to_string(index=False)); print(freq.round(2).to_string())

with pd.ExcelWriter(os.path.join(REV_OUT, "meaning_search_results.xlsx")) as xw:
    M6.to_excel(xw, sheet_name="Means"); PC6.to_excel(xw, sheet_name="Paired", index=False)
    S6.to_excel(xw, sheet_name="Stability_overlap", index=False); freq.to_frame("selection frequency").to_excel(xw, sheet_name="Feature_frequency")
    allr.to_excel(xw, sheet_name="All_runs", index=False)
    pd.DataFrame([{"run": r + 1, "method": m, "features": ", ".join(v)} for r in sorted(st6) for m, v in st6[r]["subsets"].items()]).to_excel(xw, sheet_name="Subsets", index=False)
print("Saved to", os.path.join(REV_OUT, "meaning_search_results.xlsx"))

                                             n_features  macro_F1  F1_gauss_10  F1_semantic_gauss_10
method                                                                                              
Proposed (range-coded search, paper)               20.5    0.7896       0.7165                0.7201
Proposed (meaning-coded search)                    18.9    0.7836       0.6991                0.7164
All features                                       41.0    0.7945       0.7106                0.7126
RFE (top-k), k matched                             18.9    0.7926       0.6983                0.7083
NSGA-II (3-obj), range-coded search (paper)        13.6    0.7856       0.6835                0.6929
NSGA-II (3-obj), meaning-coded search              13.4    0.7763       0.6732                0.6884
MI (top-k), k matched                              18.9    0.7856       0.6840                0.6834
              metric                                          vs  mean diff  95% CI low  95